In [ ]:
!pip install -q transformers accelerate sentence-transformers chromadb pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

In [ ]:
import os
import json
import pandas as pd
import chromadb
import torch

from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

In [ ]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_files = list(uploaded.keys())

print("Uploaded files:")
for filename in pdf_files:
    print("-", filename)

Saving Spotify-20-F-Filing.pdf to Spotify-20-F-Filing.pdf
Uploaded files:
- Spotify-20-F-Filing.pdf


In [ ]:
from pypdf import PdfReader

pdf_path = pdf_files[0]

reader = PdfReader(pdf_path)

print("Number of pages:", len(reader.pages))

Number of pages: 407


In [ ]:
pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    pages.append({
        "page": page_number,
        "text": text
    })

print(pages[0]["text"][:1000])

UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
___________________________________________________________
FORM 20-F
___________________________________________________________
☐ REGISTRATION STATEMENT PURSUANT TO SECTION 12(b) OR 12(g) OF THE SECURITIES EXCHANGE ACT OF 1934
OR
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended December 31, 2025
OR
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
OR
☐ SHELL COMPANY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
Commission File Number: 001-38438
Spotify Technology S.A.
(Exact name of Registrant as specified in its charter)
___________________________________________________________
Grand Duchy of Luxembourg
(Jurisdiction of incorporation)
33 Boulevard Prince Henri
L-1724 Luxembourg
Grand Duchy of Luxembourg
(Address of principal executive offices)
________________________________

In [ ]:
def chunk_text(text, chunk_size=700, overlap=100):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

        start += chunk_size - overlap

    return chunks

In [ ]:
sample_text = pages[0]["text"]

chunks = chunk_text(sample_text)

print("Number of chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0])

Number of chunks: 7

First chunk:

UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
___________________________________________________________
FORM 20-F
___________________________________________________________
☐ REGISTRATION STATEMENT PURSUANT TO SECTION 12(b) OR 12(g) OF THE SECURITIES EXCHANGE ACT OF 1934
OR
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended December 31, 2025
OR
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
OR
☐ SHELL COMPANY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
Commission File Number: 001-38438
Spotify Technology S.A.
(Exact name of R


In [ ]:
all_chunks = []

for page_data in pages:

    page_chunks = chunk_text(
        page_data["text"],
        chunk_size=700,
        overlap=100
    )

    for chunk_number, chunk in enumerate(page_chunks):

        all_chunks.append({
            "text": chunk,
            "page": page_data["page"],
            "chunk": chunk_number,
            "source": pdf_path
        })

print("Total chunks:", len(all_chunks))

Total chunks: 1910


In [ ]:
for item in all_chunks[:3]:
    print("Source:", item["source"])
    print("Page:", item["page"])
    print("Chunk:", item["chunk"])
    print(item["text"][:500])
    print("-" * 80)

Source: Spotify-20-F-Filing.pdf
Page: 1
Chunk: 0
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
___________________________________________________________
FORM 20-F
___________________________________________________________
☐ REGISTRATION STATEMENT PURSUANT TO SECTION 12(b) OR 12(g) OF THE SECURITIES EXCHANGE ACT OF 1934
OR
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended December 31, 2025
OR
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SEC
--------------------------------------------------------------------------------
Source: Spotify-20-F-Filing.pdf
Page: 1
Chunk: 1
TIES EXCHANGE ACT OF 1934
Commission File Number: 001-38438
Spotify Technology S.A.
(Exact name of Registrant as specified in its charter)
___________________________________________________________
Grand Duchy of Luxembourg
(Jurisdiction of incorporation)
33 Boulevard Prince Henri
L-1724 Luxembourg
Grand Duchy of Luxe

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
sentence = "Retrieval-Augmented Generation uses external documents."

embedding = embedding_model.encode(sentence)

print("Embedding type:", type(embedding))
print("Embedding dimensions:", len(embedding))

Embedding type: <class 'numpy.ndarray'>
Embedding dimensions: 384


In [ ]:
texts = [item["text"] for item in all_chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("Number of embeddings:", len(embeddings))
print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

Number of embeddings: 1910
Embedding shape: (1910, 384)


In [ ]:
import chromadb

client = chromadb.PersistentClient(
    path="./chroma_db"
)

In [ ]:
collection = client.get_or_create_collection(
    name="rag_documents"
)

In [ ]:
ids = [
    f"{item['source']}_page_{item['page']}_chunk_{item['chunk']}"
    for item in all_chunks
]

In [ ]:
metadatas = [
    {
        "source": item["source"],
        "page": item["page"],
        "chunk": item["chunk"]
    }
    for item in all_chunks
]

In [ ]:
collection.upsert(
    ids=ids,
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

In [ ]:
print("Documents in collection:", collection.count())

Documents in collection: 1910


In [ ]:
result = collection.get(
    limit=3,
    include=["documents", "metadatas"]
)

for document, metadata in zip(
    result["documents"],
    result["metadatas"]
):
    print(metadata)
    print(document[:300])
    print("-" * 80)

{'page': 1, 'source': 'Spotify-20-F-Filing.pdf', 'chunk': 0}
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
___________________________________________________________
FORM 20-F
___________________________________________________________
☐ REGISTRATION STATEMENT PURSUANT TO SECTION 12(b) OR 12(g) OF THE SECURITIES EXCHANGE ACT OF 1934
--------------------------------------------------------------------------------
{'page': 1, 'source': 'Spotify-20-F-Filing.pdf', 'chunk': 1}
TIES EXCHANGE ACT OF 1934
Commission File Number: 001-38438
Spotify Technology S.A.
(Exact name of Registrant as specified in its charter)
___________________________________________________________
Grand Duchy of Luxembourg
(Jurisdiction of incorporation)
33 Boulevard Prince Henri
L-1724 Luxembourg
--------------------------------------------------------------------------------
{'chunk': 2, 'source': 'Spotify-20-F-Filing.pdf', 'page': 1}
______________________________________
Securities reg

In [ ]:
question = "What does the document say about the main system requirements?"

In [ ]:
question_embedding = embedding_model.encode(
    question
)

In [ ]:
results = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=4
)

In [ ]:
for i, document in enumerate(results["documents"][0]):

    print(f"Result {i + 1}")
    print("Source:", results["metadatas"][0][i])
    print(document)
    print("-" * 80)

Result 1
Source: {'page': 6, 'source': 'Spotify-20-F-Filing.pdf', 'chunk': 5}
rdware, networks, regulations, and standards that we do not control;
• our ability to maintain the integrity of our technology infrastructure and systems or the security of confidential information;
• undetected errors, misconfigurations, bugs, or vulnerabilities in our products and services;
3
--------------------------------------------------------------------------------
Result 2
Source: {'source': 'Spotify-20-F-Filing.pdf', 'chunk': 8, 'page': 16}
e certain licenses and royalty rates;
• calculate and make payments based on complex royalty structures, which requires tracking usage of content on our Service that may have inaccurate or incomplete metadata
necessary for such calculation;
• provide periodic reports on the exploitation of the content;
• provide advertising inventory at discounted rates or on other favorable terms;
• comply with certain service offering restrictions;
• comply with certain market

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [ ]:
print(model.device)

cuda:0


In [ ]:
def generate_answer(prompt, max_new_tokens=200):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [ ]:
prompt = """
Explain the purpose of Retrieval-Augmented Generation
in two or three sentences.
"""

print(generate_answer(prompt))

Retrieval-Augmented Generation is a technique that combines the strengths of both retrieval and generation models to improve text completion tasks. It leverages pre-trained language models to generate new text while also using them as a source for generating more relevant completions based on previous inputs, thus enhancing the quality and diversity of generated outputs.

In essence, it allows these models to learn from their own past interactions with data, making them better at understanding context and predicting future events accurately. This approach can be particularly useful in applications where high-quality, diverse, and contextually appropriate responses are crucial, such as chatbots, virtual assistants, and content creation tools. By combining the power of both types of models, RAG aims to create systems that not only produce human-like output but also adapt to user needs over time through continuous learning and feedback loops. 

This method has been shown to significantly 

In [ ]:
def build_context(results):

    context_parts = []

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    for document, metadata in zip(documents, metadatas):

        source = metadata["source"]
        page = metadata["page"]

        context_parts.append(
            f"[Source: {source}, Page: {page}]\n"
            f"{document}"
        )

    return "\n\n".join(context_parts)

In [ ]:
def build_rag_prompt(question, context):

    return f"""
You are answering a question using the provided document context.

Rules:
1. Use the provided context as the primary source of information.
2. Do not invent facts that are not supported by the context.
3. If the context does not contain enough information, say:
   "I don't know based on the provided documents."
4. Give a concise and direct answer.

DOCUMENT CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

In [ ]:
def rag_answer(
    question,
    n_results=4,
    max_new_tokens=200
):

    # 1. Embed the question
    question_embedding = embedding_model.encode(
        question
    )

    # 2. Retrieve relevant chunks
    results = collection.query(
        query_embeddings=[
            question_embedding.tolist()
        ],
        n_results=n_results
    )

    # 3. Build context
    context = build_context(results)

    # 4. Build prompt
    prompt = build_rag_prompt(
        question,
        context
    )

    # 5. Generate answer
    answer = generate_answer(
        prompt,
        max_new_tokens=max_new_tokens
    )

    return {
        "question": question,
        "answer": answer,
        "context": context,
        "results": results
    }

In [ ]:
question = "What are the risks of investing in spotify"

result = rag_answer(question)

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

QUESTION:
What are the risks of investing in spotify

ANSWER:
The risks associated with investing in Spotify include:

1. The company's ability to generate revenue from its streaming services, which depends on factors like user engagement and content quality.
2. Competition from other music streaming platforms, which could lead to declining subscriber numbers and revenue.
3. Changes in consumer preferences towards alternative forms of entertainment, affecting the demand for Spotify's service.
4. Legal challenges related to copyright infringement claims, which could result in fines or legal action against the company.
5. Financial commitments required under certain license agreements, which may limit the company's operational flexibility and negatively impact its business, operating results, and financial condition.
6. Potential conflicts of interest due to the company's ownership structure, where the board members may be influenced by their personal interests rather than the best inter

In [ ]:
for i, metadata in enumerate(
    result["results"]["metadatas"][0]
):

    print(
        f"Retrieved source {i + 1}: "
        f"{metadata['source']}, "
        f"page {metadata['page']}, "
        f"chunk {metadata['chunk']}"
    )

Retrieved source 1: Spotify-20-F-Filing.pdf, page 351, chunk 4
Retrieved source 2: Spotify-20-F-Filing.pdf, page 16, chunk 8
Retrieved source 3: Spotify-20-F-Filing.pdf, page 34, chunk 7
Retrieved source 4: Spotify-20-F-Filing.pdf, page 311, chunk 4


In [ ]:
questions = [
    "What is the main purpose of the system?",
    "What are the important requirements?",
    "What process does the document describe?"
]

for question in questions:

    result = rag_answer(question)

    print("=" * 80)
    print("QUESTION:", question)
    print("ANSWER:", result["answer"])

QUESTION: What is the main purpose of the system?
ANSWER: The main purpose of the system is to ensure uninterrupted service delivery, which includes maintaining robust infrastructure and ensuring compliance with various legal and regulatory requirements. This involves leveraging external resources like Google Cloud Platform (GCP) for critical functions such as data processing, storage, and computing capabilities. The reliance on GCP reflects the company's strategy to offload complex tasks onto third-party providers while still managing core aspects of their technology stack internally. This approach helps mitigate potential disruptions caused by issues within their own systems or those of third-party vendors.
QUESTION: What are the important requirements?
ANSWER: The important requirements include:

- Calculating and making payments based on complex royalty structures, requiring accurate metadata tracking.
- Providing periodic reports on content exploitation.
- Offering advertising inv

In [ ]:
question = "What is the population of Mars according to this document?"

result = rag_answer(question)

print(result["answer"])

I don't know based on the provided documents. The document only provides data related to employee numbers at Spotify, including their geographical distribution across different locations such as United States, Sweden, and United Kingdom. It does not mention any information about the population of Mars or provide any relevant data from that topic. Therefore, I cannot determine the population of Mars from the given information.


In [ ]:
def load_text_file(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return f.read()

In [ ]:
def load_json_file(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    return json.dumps(
        data,
        indent=2,
        ensure_ascii=False
    )

In [ ]:
def load_csv_file(path):

    df = pd.read_csv(path)

    return df.to_string(
        index=False
    )

In [ ]:
retrieval_values = [1, 2, 4, 6, 8]

In [ ]:
for k in retrieval_values:

    result = rag_answer(
        "What is the main purpose of the system?",
        n_results=k
    )

    print("=" * 80)
    print("Retrieved chunks:", k)
    print(result["answer"])

Retrieved chunks: 1
The main purpose of the system appears to be managing or controlling various aspects such as system items and processes effectively. However, without more specific details from the document, I can't provide a more precise description of what exactly the system aims to achieve. The text suggests it's focused on ensuring that all elements related to systems (items and processes) function well. 

Given this limited information, my best guess for the main purpose would be:

To manage and control system items and processes effectively. 

This interpretation aligns with the phrase "appropriate and effective" mentioned in the context, which implies that the system is designed to handle its tasks efficiently and correctly. 

Please note that this conclusion is based solely on the information provided in the given context and may not reflect the full scope of the system's objectives. For a comprehensive understanding, additional details from the actual document would be nece

# 22. Experiment: Changing Chunk Size

Create alternative configurations:

```python
chunk_sizes = [
    (300, 50),
    (700, 100),
    (1200, 150)
]
```

| Chunk configuration | Retrieval quality | Context completeness | Prompt size |
| ------------------- | ----------------- | -------------------- | ----------- |
| 300 / 50            |                   |                      |       1781      |
| 700 / 100           |                   |                      |     1434        |
| 1200 / 150          |                   |                      |      2334       |


In [ ]:
chunk_sizes = [
    (300, 50),
    (700, 100),
    (1200, 150)
]

question = "What is the main purpose of the system?"

for i, (chunk_size, overlap) in enumerate(chunk_sizes):
    print(f"\n{'='*80}")
    print(f"EXPERIMENT {i+1}: Chunk Size = {chunk_size}, Overlap = {overlap}")
    print(f"{'='*80}\n")

    # 1. Re-chunk text
    current_all_chunks = []
    for page_data in pages:
        page_chunks = chunk_text(
            page_data["text"],
            chunk_size=chunk_size,
            overlap=overlap
        )
        for chunk_number, chunk in enumerate(page_chunks):
            current_all_chunks.append({
                "text": chunk,
                "page": page_data["page"],
                "chunk": chunk_number,
                "source": pdf_path
            })

    # 2. Prepare for embedding and upserting
    current_ids = [
        f"{item['source']}_page_{item['page']}_chunk_{item['chunk']}"
        for item in current_all_chunks
    ]
    current_texts = [item["text"] for item in current_all_chunks]
    current_metadatas = [
        {
            "source": item["source"],
            "page": item["page"],
            "chunk": item["chunk"]
        }
        for item in current_all_chunks
    ]

    # 3. Generate new embeddings
    print(f"Generating {len(current_texts)} embeddings for chunk size {chunk_size}...")
    current_embeddings = embedding_model.encode(
        current_texts,
        show_progress_bar=False
    )
    print("Embeddings generated.")

    # 4. Clear existing collection and upsert new data
    # IMPORTANT: This clears the collection for each iteration for a clean experiment.
    # Remove the existing collection if it exists
    try:
        client.delete_collection(name="rag_documents")
        print("Existing collection 'rag_documents' deleted.")
    except Exception as e:
        print(f"No existing collection 'rag_documents' to delete, or an error occurred: {e}")

    # Recreate the collection
    collection = client.get_or_create_collection(name="rag_documents")

    collection.upsert(
        ids=current_ids,
        documents=current_texts,
        embeddings=current_embeddings.tolist(),
        metadatas=current_metadatas
    )
    print(f"Collection updated with {collection.count()} documents for chunk size {chunk_size}")

    # 5. Run RAG query
    rag_result = rag_answer(question)

    print("\nQUESTION:")
    print(rag_result["question"])

    print("\nANSWER:")
    print(rag_result["answer"])

    print("\nRETRIEVED CONTEXT:")
    # Print a summary of the context to keep output manageable
    context_summary = rag_result["context"][:500] + ("..." if len(rag_result["context"]) > 500 else "")
    print(context_summary)

    # Calculate prompt size
    rag_prompt = build_rag_prompt(question, rag_result["context"])
    print(f"\nPROMPT SIZE (characters): {len(rag_prompt)}")

    print(f"\n>>> Please manually evaluate Retrieval Quality and Context Completeness based on the Answer and Retrieved Context for Chunk Size = {chunk_size}, Overlap = {overlap}. <<<")


EXPERIMENT 1: Chunk Size = 300, Overlap = 50

Generating 3453 embeddings for chunk size 300...
Embeddings generated.
Existing collection 'rag_documents' deleted.
Collection updated with 3453 documents for chunk size 300

QUESTION:
What is the main purpose of the system?

ANSWER:
The main purpose of the system is to combine various elements such as hardware, software, equipment, facilities, personnel, processes, and procedures to achieve the capability required to fulfill a specific need. This involves ensuring that all necessary components are available and functioning properly before proceeding with production. Additionally, it includes identifying opportunities to gather unique, non-primary mission-related data early in the development process to enhance overall system effectiveness.

RETRIEVED CONTEXT:
[Source: 20170001761.pdf, Page: 16]
rement of a system. A “system” is 
the combination of elements that function together to produce the capability required to meet a 
need. The elem

# 24. Experiment: Base Qwen vs. Qwen + RAG

Run the same question using both configurations.

```python
question = "What does the document say about the system requirements?"

base_prompt = f"""
Answer the following question:

{question}
"""

base_answer = generate_answer(base_prompt)

rag_result = rag_answer(question)

print("BASE MODEL:")
print(base_answer)

print("\nRAG:")
print(rag_result["answer"])
```

Record the results.

| Configuration | Correct? | Grounded? | Useful? |
| ------------- | -------: | --------: | ------: |
| Qwen          |          |           |         |
| Qwen + RAG    |          |           |         |